## 0.1 Init ambiente Google Colab

Conecto la virtual machine de Google Colab con el Google Drive para tener persistencia de archivos.

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Para correr la siguiente celda es fundamental, POR UNICA VEZ, seguir estos pasos:

* Registrar usuario en Kaggle con la cuenta de email de la Universidad Austral
* Hacer el "Join Competition" a la competencia de Labo 3
* Generar el archivo kaggle.json en https://www.kaggle.com/settings/account ("Create Legacy API Key")
* Crear carpeta labo3 en el Google Drive, y dentro una carpeta kaggle
* Subir kaggle.json a esa carpeta kaggle

In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"

# z102 — EDA orientado a la decisión de modelado

Este es mi análisis exploratorio para la competencia. Uso el mismo stack que el EDA base
(`z101`: **DuckDB + plotly**), pero lo reoriento: no busco describir todo el dataset, busco
**decidir dónde invertir el esfuerzo de modelado y qué features construir** para los modelos
que voy a usar (LightGBM / Deep Learning), dejando de paso a AutoARIMA (`z251`) el subconjunto
que ese modelo puede aprovechar.

**El problema.** "La Multinacional" (Unilever) hace forecasting **mensual** de sell-in de
consumo masivo. Debo predecir las `tn` de **202002** para los **780 productos** de
`product_id_apredecir201912`.

**Sobre el horizonte +2.** La historia llega hasta **201912**, así que 202002 es el horizonte
**mes+2** y se saltea 202001. Esto es una **decisión operativa** de la competencia (el mes
inmediato siguiente ya está comprometido en producción; lo accionable es el +2), y **no** debe
confundirse con el lag físico de ~2 meses entre el sell-in (sale el camión de planta) y el
sell-out (se escanea en la góndola). Ese lag físico lo uso solo para **interpretar la estacionalidad**.

Las conclusiones consolidadas (los 5 hallazgos clave) están al final, en la **§8**.

In [ ]:
import os as os
import duckdb
import numpy as np
import pandas as pd
import plotly.express as px

In [ ]:
PARAM = {'experimento':'EDA-102', 'kaggle_competition':'labo-iii-2026-ba'}

In [ ]:
ruta = "/content/buckets/b1/" + PARAM['experimento']
print(ruta); os.makedirs(ruta, exist_ok=True); os.chdir(ruta)

## 1 Carga de tablas

Cargo a DuckDB. Sumo dos tablas que el EDA base descargaba pero no usaba:
`tb_apredecir` (los 780 productos objetivo) y `tb_stocks`.

In [ ]:
con = duckdb.connect()

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE tb_sellin AS
    SELECT CAST(strptime(CAST( periodo AS VARCHAR), '%Y%m') AS DATE) periodo,
           customer_id, product_id, plan_precios_cuidados,
           cust_request_qty, cust_request_tn, tn
    FROM read_csv_auto('/content/datasets/sell-in.txt.gz')
    ORDER BY product_id, periodo, customer_id
""")

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE tb_productos AS
    SELECT * FROM read_csv_auto('/content/datasets/tb_productos.txt') ORDER BY product_id
""")

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE tb_apredecir AS
    SELECT * FROM read_csv_auto('/content/datasets/product_id_apredecir201912.txt')
""")
print("Productos a predecir:", con.sql("SELECT COUNT(*) FROM tb_apredecir").fetchone()[0])

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE tb_stocks AS
    SELECT * FROM read_csv_auto('/content/datasets/tb_stocks.txt')
""")

In [ ]:
print("=== tb_productos ==="); display(con.sql("DESCRIBE tb_productos").df())
print("=== tb_stocks ===");    display(con.sql("DESCRIBE tb_stocks").df())

In [ ]:
display(con.sql("""
    SELECT MIN(periodo) primer_periodo, MAX(periodo) ultimo_periodo,
           COUNT(DISTINCT periodo) n_periodos, COUNT(DISTINCT product_id) n_productos,
           COUNT(DISTINCT customer_id) n_clientes, COUNT(*) n_filas
    FROM tb_sellin
""").df())
PRIMER = con.sql("SELECT MIN(periodo) FROM tb_sellin").fetchone()[0]
ULT    = con.sql("SELECT MAX(periodo) FROM tb_sellin").fetchone()[0]
print("Rango:", PRIMER, "->", ULT, "| objetivo = 202002 (mes+2)")

## 2 La métrica manda: el error está ponderado por volumen

Arranco por acá porque condiciona todo lo demás. La nota sale del **Total Forecast Error**,
en esencia un error agregado **ponderado por toneladas**:

$$\text{TFE} = \frac{\sum_{p} \lvert \hat{y}_p - y_p \rvert}{\sum_{p} y_p}$$

*(Pendiente: confirmar la fórmula exacta en Kaggle.)* Como el denominador es constante,
minimizar TFE equivale a minimizar el **error absoluto total en toneladas** → lo dominan los
productos de mayor volumen. Por eso lo primero es ver **dónde está el volumen objetivo**. Uso
el volumen 2019 como proxy del peso de cada producto.

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE tb_peso AS
    SELECT a.product_id, COALESCE(SUM(s.tn), 0) tn_2019
    FROM tb_apredecir a
    LEFT JOIN tb_sellin s
      ON s.product_id = a.product_id
     AND s.periodo BETWEEN DATE '2019-01-01' AND DATE '2019-12-01'
    GROUP BY a.product_id
""")
df = con.sql("SELECT product_id, tn_2019, 100.0*tn_2019/SUM(tn_2019) OVER () pct FROM tb_peso ORDER BY tn_2019 DESC").df()
df["pct_acum"] = df["pct"].cumsum()
for u in [50, 80, 95]:
    n = int((df["pct_acum"] <= u).sum()) + 1
    print(f"{n:4d} productos ({100*n/len(df):4.1f}%) -> {u}% del volumen objetivo")
print(f"{int((df['tn_2019']==0).sum())} productos objetivo SIN ventas 2019")
display(df.head(15))

In [ ]:
df_p = df.reset_index(drop=True); df_p["rank"] = np.arange(1, len(df_p)+1)
gra = px.line(df_p, x="rank", y="pct_acum",
              title="§2 Concentración del volumen objetivo (Pareto de 780 productos)",
              labels={"rank":"productos ordenados por volumen","pct_acum":"% acumulado"})
gra.add_hline(y=80, line_dash="dash"); gra.show()

**Decisión §2.** El volumen está muy concentrado: priorizo precisión en los high-volume y
vigilo mi error ahí por encima del promedio simple. La cola larga no mueve la nota.

## 3 Estructura del portfolio: dónde está el volumen real

Detecto dinámicamente las columnas de categoría (las que empiezan con `cat`) y uso la más gruesa
(`cat1`) como unidad de negocio.

In [ ]:
prod_cols = con.sql("DESCRIBE tb_productos").df()["column_name"].tolist()
cat_cols = [c for c in prod_cols if c.lower().startswith("cat")]
CAT_BU = cat_cols[0] if cat_cols else "cat1"
print("Columnas de categoría:", cat_cols, "| unidad de negocio:", CAT_BU)

In [ ]:
df = con.sql(f"""SELECT p.{CAT_BU} unidad, SUM(s.tn) tn
    FROM tb_sellin s JOIN tb_productos p USING (product_id)
    WHERE s.periodo BETWEEN DATE '2019-01-01' AND DATE '2019-12-01'
    GROUP BY 1 ORDER BY tn DESC""").df()
df["pct"] = 100*df["tn"]/df["tn"].sum()
display(df)
px.bar(df, x="unidad", y="pct", title=f"§3 Share de volumen 2019 por unidad de negocio ({CAT_BU})",
       labels={"unidad":"unidad","pct":"% del volumen"}).show()

In [ ]:
df = con.sql(f"""SELECT p.{CAT_BU} unidad, SUM(pe.tn_2019) tn
    FROM tb_peso pe JOIN tb_productos p USING (product_id) GROUP BY 1 ORDER BY tn DESC""").df()
df["pct"] = 100*df["tn"]/df["tn"].sum()
display(df)
px.bar(df, x="unidad", y="pct", title="§3 Share del volumen OBJETIVO (780) por unidad de negocio",
       labels={"unidad":"unidad","pct":"% del volumen objetivo"}).show()

**Lectura §3.** El volumen se concentra en **HC (Home Care/limpieza, ~60%)**; PC y FOODS rondan
20% cada uno y REF es residual. El feature engineering y el tuning tienen que ir a HC.

## 4 Ciclo de vida del producto (ramp-up y discontinuación)

Solo tengo **sell-in**, no sell-out. El arranque (ramp-up) y el apagado (discontinuación) no son
demanda estable: en el ramp-up la serie crece mientras se llena el canal, y en la discontinuación
cae a cero por decisión comercial. Los detecto y aíslo.

**Nota metodológica (left-censoring).** La regla "primeros 3 meses = ramp-up" se aplica **solo a
productos cuyo `mes_alta` es posterior al inicio del dataset**. Un producto que ya vendía en el
primer período (201701) no es un lanzamiento nuevo: simplemente la data empieza ahí, y marcar sus
primeros 3 meses como ramp-up borraría meses válidos de productos maduros (incluidos los grandes).

In [ ]:
con.execute("""CREATE OR REPLACE TABLE tb_ventas_pm AS
    SELECT product_id, periodo, SUM(tn) tn FROM tb_sellin GROUP BY 1,2 ORDER BY 1,2""")
display(con.sql("SELECT * FROM tb_ventas_pm LIMIT 5").df())

In [ ]:
con.execute(f"""CREATE OR REPLACE TABLE tb_vida AS
    SELECT product_id, MIN(periodo) mes_alta, MAX(periodo) mes_baja,
           COUNT(*) FILTER (WHERE tn > 0) meses_con_venta,
           datediff('month', MIN(periodo), MAX(periodo)) + 1 AS span_meses,
           (MAX(periodo) < DATE '{ULT}') AS discontinuado
    FROM tb_ventas_pm GROUP BY product_id""")
display(con.sql("SELECT * FROM tb_vida ORDER BY product_id LIMIT 10").df())

In [ ]:
px.histogram(con.sql("SELECT span_meses FROM tb_vida").df(), x="span_meses", nbins=36,
             title="§4 Meses de vida por producto", labels={"span_meses":"meses de vida"}).show()
display(con.sql("SELECT discontinuado, COUNT(*) n FROM tb_vida GROUP BY 1 ORDER BY 1").df())

Cuánta historia tienen los **780 objetivo**: explica los productos que en `z251` no permiten
ajustar estacionalidad anual (caen a la 2ª pasada de AutoARIMA).

In [ ]:
display(con.sql(f"""SELECT COUNT(*) total,
   SUM((v.mes_baja>=DATE '{ULT}')::INT) activos, SUM(v.discontinuado::INT) discont,
   SUM((v.span_meses<12)::INT) menos12m, SUM((v.span_meses<24)::INT) menos24m
   FROM tb_apredecir a JOIN tb_vida v USING(product_id)""").df())

In [ ]:
# panel con flags de ciclo de vida.  es_ramp_up NO aplica a productos presentes desde el inicio.
con.execute(f"""CREATE OR REPLACE TABLE tb_panel AS
    SELECT v.product_id, v.periodo, v.tn,
           datediff('month', d.mes_alta, v.periodo) AS meses_desde_lanzamiento,
           ( datediff('month', d.mes_alta, v.periodo) < 3
             AND d.mes_alta > DATE '{PRIMER}' )                                AS es_ramp_up,
           ( d.discontinuado AND datediff('month', v.periodo, d.mes_baja) <= 2 ) AS es_discontinuacion
    FROM tb_ventas_pm v JOIN tb_vida d USING (product_id)""")
display(con.sql("""SELECT SUM(es_ramp_up::INT) ramp, SUM(es_discontinuacion::INT) disc, COUNT(*) tot FROM tb_panel""").df())

In [ ]:
def graficar_ciclo_vida(producto):
  df = con.sql(f"""SELECT periodo, tn,
        CASE WHEN es_ramp_up THEN 'ramp-up (<3m)'
             WHEN es_discontinuacion THEN 'discontinuación' ELSE 'ventana estable' END AS tramo
        FROM tb_panel WHERE product_id = {producto} ORDER BY periodo""").df()
  t = con.sql(f"SELECT CONCAT_WS(', ', *COLUMNS('.*')) FROM tb_productos WHERE product_id={producto}").fetchone()[0]
  g = px.scatter(df, x="periodo", y="tn", color="tramo", title=f"§4 [{producto}] {t}")
  g.update_traces(mode='markers+lines'); g.update_yaxes(rangemode="tozero"); g.show()

for p in [20001, 20494, 20554]: graficar_ciclo_vida(p)

**Decisión §4.** Quedan `meses_desde_lanzamiento`, `es_ramp_up` y `es_discontinuacion`.
ARIMA: filtro esas filas. GBM/DL: no las tiro, `meses_desde_lanzamiento` entra como feature.

## 5 Estacionalidad por categoría (re-evaluada por volumen)

Mido un **índice estacional** = `tn` promedio del mes / `tn` promedio de la categoría, sobre la
ventana estable, y lo cruzo con §3 para ver si la estacionalidad fuerte coincide con bajo volumen.

In [ ]:
dfm = con.sql("SELECT month(periodo) mes, AVG(tn) tnm FROM tb_panel WHERE NOT es_ramp_up AND NOT es_discontinuacion GROUP BY 1 ORDER BY 1").df()
px.bar(dfm, x="mes", y="tnm", title="§5 Perfil estacional global (promedio por mes)").update_xaxes(dtick=1).show()

In [ ]:
df = con.sql(f"""SELECT p.{CAT_BU} cat, month(v.periodo) mes, v.tn
    FROM tb_panel v JOIN tb_productos p USING(product_id)
    WHERE NOT v.es_ramp_up AND NOT v.es_discontinuacion""").df()
top = df.groupby("cat")["tn"].sum().sort_values(ascending=False).head(12).index
df = df[df["cat"].isin(top)]
piv = df.groupby(["cat","mes"])["tn"].mean().reset_index().pivot(index="cat", columns="mes", values="tn")
ind = piv.div(piv.mean(axis=1), axis=0)
px.imshow(ind, aspect="auto", color_continuous_scale="RdBu_r", origin="upper",
          title="§5 Índice estacional por categoría (cat1) (1.0 = mes promedio)",
          labels={"x":"mes","y":"categoría","color":"índice"}).update_xaxes(dtick=1).show()
display(ind.round(2))

**Lectura §5.** La estacionalidad fuerte vive en las categorías de **menor volumen** (REF, FOODS);
HC (60% del volumen) es más plano. **Ojo**: en octubre suben las tres grandes, y en toneladas
absolutas HC es el que más aporta al pico (ver §6). Para GBM/DL: dummies de mes + índice estacional
por categoría, sin sobre-invertir en Foods.

## 6 Pico de cierre de año (forzado de canal)

Observo un patrón recurrente de fin de año: pico seguido de caída, consistente con **carga de canal**
(se empuja sell-in para cerrar el año, inflando por encima de la demanda real). Lo verifico y, si es
sistemático, lo aíslo con un flag binario.

In [ ]:
# índice mensual con octubre resaltado (más legible que la línea)
df = con.sql("SELECT month(periodo) mes, SUM(tn) tn FROM tb_panel WHERE NOT es_ramp_up AND NOT es_discontinuacion GROUP BY 1 ORDER BY 1").df()
df["indice"] = df["tn"] / df["tn"].mean()
df["mes_nombre"] = ["Ene","Feb","Mar","Abr","May","Jun","Jul","Ago","Sep","Oct","Nov","Dic"]
df["color"] = ["Octubre" if m == 10 else "Resto" for m in df["mes"]]
gra = px.bar(df, x="mes_nombre", y="indice", color="color",
             color_discrete_map={"Octubre":"#e3522f","Resto":"#9aa4b2"},
             text=df["indice"].round(2),
             title="§6 Estacionalidad del sell-in: el pico de octubre (push de cierre de año)",
             labels={"mes_nombre":"mes","indice":"índice (1.0 = mes promedio)"})
gra.add_hline(y=1.0, line_dash="dash"); gra.update_layout(showlegend=False); gra.show()

In [ ]:
# verifico que octubre se repite TODOS los años (no es un artefacto del promedio)
dfa = con.sql("SELECT year(periodo) anio, month(periodo) mes, SUM(tn) tn FROM tb_panel WHERE NOT es_ramp_up AND NOT es_discontinuacion GROUP BY 1,2").df()
dfa["indice"] = dfa.groupby("anio")["tn"].transform(lambda x: x/x.mean())
dfa["anio"] = dfa["anio"].astype(str)
g = px.line(dfa, x="mes", y="indice", color="anio", markers=True,
            title="§6 Índice mensual por año: octubre se repite; el bump de otoño (may) se corre")
g.add_hline(y=1.0, line_dash="dash"); g.update_xaxes(dtick=1); g.show()

In [ ]:
# creo el feature: flag binario del mes del pico de cierre (octubre, validado arriba)
MES_PICO = 10
con.execute(f"""CREATE OR REPLACE TABLE tb_panel AS
    SELECT *, (month(periodo) = {MES_PICO})::INT AS flag_cierre_anio FROM tb_panel""")
display(con.sql("SELECT flag_cierre_anio, COUNT(*) n, ROUND(AVG(tn),2) tn_medio FROM tb_panel WHERE NOT es_ramp_up AND NOT es_discontinuacion GROUP BY 1").df())

**Decisión §6.** Octubre es el máximo del año (+24%) y cae después; **se repite los 3 años**
→ señal real. Queda `flag_cierre_anio`. El bump de mayo **no** se toma como señal: cuando separás por
año se desarma (en 2018 picó en marzo), es un artefacto de promediar. ARIMA: exógena `X=`. GBM/DL:
flag binario (+ `post_cierre` para la caída nov-dic). El de otoño lo absorben las dummies de mes.

## 7 Heterogeneidad de clientes

Hipótesis inicial: los clientes **chicos** comprarían de forma más regular (menor CV). La pongo a
prueba con `CV = desvío/media` de la `tn` mensual por cliente. (No lo usa ARIMA, que agrega clientes;
es insumo para GBM/DL.)

In [ ]:
con.execute("CREATE OR REPLACE TABLE tb_cli_mes AS SELECT customer_id, periodo, SUM(tn) tn FROM tb_sellin GROUP BY 1,2")
con.execute("""CREATE OR REPLACE TABLE tb_clientes AS
    SELECT customer_id, SUM(tn) total_tn, COUNT(*) n_meses, AVG(tn) tn_medio,
           STDDEV_SAMP(tn)/NULLIF(AVG(tn),0) cv FROM tb_cli_mes GROUP BY 1""")
display(con.sql("SELECT * FROM tb_clientes ORDER BY total_tn DESC LIMIT 5").df())

In [ ]:
df = con.sql("SELECT * FROM tb_clientes WHERE n_meses >= 6").df()
px.scatter(df, x="total_tn", y="cv", opacity=0.4, trendline="lowess",
           title="§7 CV vs volumen del cliente", labels={"total_tn":"volumen (tn)","cv":"CV"}).update_xaxes(type="log").show()

In [ ]:
df = con.sql("SELECT * FROM tb_clientes WHERE n_meses >= 6").df()
df["decil"] = pd.qcut(df["total_tn"], 10, labels=False) + 1
res = df.groupby("decil").agg(cv_mediano=("cv","median"), n=("customer_id","count")).reset_index()
display(res)
px.bar(res, x="decil", y="cv_mediano", title="§7 CV mediano por decil de tamaño (1=chico,10=grande)").update_xaxes(dtick=1).show()
print("corr log(volumen) vs CV:", round(np.corrcoef(np.log(df["total_tn"]), df["cv"])[0,1], 3))

**Hipótesis REFUTADA §7.** Es al revés: el CV **baja** con el volumen (corr ≈ −0.58). Los clientes
**grandes** son los regulares (ordenan parejo); los **chicos** son erráticos/lumpy. Igual sirve para
GBM/DL: `cv_cliente`, `decil_tamano`, y a nivel producto la fracción de demanda de clientes estables.

# 8 Conclusiones — los 5 hallazgos clave

Resumen ejecutivo, ordenado por impacto en la métrica. Cada hallazgo viene con su gráfico y con
"cómo lo explico" en una frase.

## Hallazgo 1 — La competencia se gana en poquísimos productos, casi todos de limpieza
**Número:** 140 productos (18%) explican el 80% del tonelaje objetivo; HC (limpieza) es ~60% del volumen.
**Por qué importa:** la métrica es en toneladas, pesa el volumen. Errar un producto grande de HC
cuesta más que errar cien chicos.
**Cómo lo explico:** *"La mitad del puntaje se juega en unos 40 productos y casi todo es limpieza;
ahí va el esfuerzo, la cola larga no mueve la nota."*

In [ ]:
df = con.sql("SELECT product_id, tn_2019, 100.0*tn_2019/SUM(tn_2019) OVER () pct FROM tb_peso ORDER BY tn_2019 DESC").df()
df = df.reset_index(drop=True); df["rank"] = np.arange(1,len(df)+1); df["pct_acum"] = df["pct"].cumsum()
g1 = px.line(df, x="rank", y="pct_acum", title="Hallazgo 1 — Concentración del volumen objetivo (Pareto)",
             labels={"rank":"productos ordenados por volumen","pct_acum":"% acumulado"})
g1.add_hline(y=80, line_dash="dash"); g1.show()

## Hallazgo 2 — El pico de octubre es carga de canal, no demanda
**Número:** octubre +24% sobre el mes promedio, y cae después; se repite los 3 años.
**Por qué importa:** es un pico artificial; si el modelo lo lee como tendencia, proyecta de más.
**Cómo lo explico:** *"Octubre siempre sube y después baja: es la empresa forzando ventas de cierre.
Lo marco con un flag para que el modelo lo entienda."*

In [ ]:
df = con.sql("SELECT month(periodo) mes, SUM(tn) tn FROM tb_panel WHERE NOT es_ramp_up AND NOT es_discontinuacion GROUP BY 1 ORDER BY 1").df()
df["indice"] = df["tn"]/df["tn"].mean()
df["mes_nombre"] = ["Ene","Feb","Mar","Abr","May","Jun","Jul","Ago","Sep","Oct","Nov","Dic"]
df["color"] = ["Octubre" if m==10 else "Resto" for m in df["mes"]]
g2 = px.bar(df, x="mes_nombre", y="indice", color="color",
            color_discrete_map={"Octubre":"#e3522f","Resto":"#9aa4b2"}, text=df["indice"].round(2),
            title="Hallazgo 2 — El pico de octubre (push de cierre de año)",
            labels={"mes_nombre":"mes","indice":"índice (1.0 = mes promedio)"})
g2.add_hline(y=1.0, line_dash="dash"); g2.update_layout(showlegend=False); g2.show()

## Hallazgo 3 — Hay que recortar el arranque y el final, y muchos objetivos tienen poca historia
**Número:** 212 de los 780 (27%) tienen <24 meses de historia → casi exactamente los ~210 que en
`z251` no ajustan estacionalidad y caen a la 2ª pasada de AutoARIMA.
**Por qué importa:** el ramp-up y la discontinuación no son demanda real; y con poca historia ningún
modelo de serie sola la pega.
**Cómo lo explico:** *"El nacimiento y la muerte del producto son logística, no demanda: los saco.
Y muchos objetivos son nuevos, con poca historia: ahí conviene un modelo global."*

In [ ]:
g3 = px.histogram(con.sql("SELECT span_meses FROM tb_vida").df(), x="span_meses", nbins=36,
                  title="Hallazgo 3 — Historia disponible por producto (meses de vida)",
                  labels={"span_meses":"meses de vida"})
g3.show()
print("De los 780 objetivo, con <24m de historia:",
      con.sql("SELECT SUM((v.span_meses<24)::INT) FROM tb_apredecir a JOIN tb_vida v USING(product_id)").fetchone()[0])

## Hallazgo 4 — La estacionalidad depende de la categoría (y cuidado con los porcentajes)
**Número:** la estacionalidad fuerte está en categorías chicas (REF, FOODS); HC es plano. Pero en
octubre, en toneladas absolutas, HC aporta el 49% del pico (Foods 33%, PC 18%).
**Por qué importa:** no conviene sobre-invertir en la estacionalidad vistosa de Foods; y mirar
índices normalizados engaña (un +38% en algo chico < un +19% en algo enorme).
**Cómo lo explico:** *"Cada categoría tiene su estacionalidad, y la más marcada es la que menos vende:
una trampa. Siempre hay que volver a las toneladas absolutas."*

In [ ]:
df = con.sql(f"""SELECT p.{CAT_BU} cat, month(v.periodo) mes, v.tn
    FROM tb_panel v JOIN tb_productos p USING(product_id)
    WHERE NOT v.es_ramp_up AND NOT v.es_discontinuacion""").df()
piv = df.groupby(["cat","mes"])["tn"].mean().reset_index().pivot(index="cat", columns="mes", values="tn")
ind = piv.div(piv.mean(axis=1), axis=0)
g4 = px.imshow(ind, aspect="auto", color_continuous_scale="RdBu_r", origin="upper",
               title="Hallazgo 4 — Índice estacional por categoría (1.0 = mes promedio)",
               labels={"x":"mes","y":"categoría","color":"índice"})
g4.update_xaxes(dtick=1); g4.show()

## Hallazgo 5 — Los clientes grandes son los predecibles, no los chicos
**Número:** CV mediano del decil chico 0.82 vs decil grande 0.37 (corr log-vol/CV ≈ −0.58).
**Por qué importa:** es lo contrario de lo intuitivo; las cadenas grandes ordenan parejo, los chicos
compran esporádico. Define qué demanda es predecible.
**Cómo lo explico:** *"Pensé que los chicos comprarían más fijo, pero es al revés: los grandes son los
regulares. Una cadena pide todos los meses lo mismo; un kiosco cuando se le antoja."*

In [ ]:
df = con.sql("SELECT * FROM tb_clientes WHERE n_meses >= 6").df()
df["decil"] = pd.qcut(df["total_tn"], 10, labels=False) + 1
res = df.groupby("decil").agg(cv_mediano=("cv","median")).reset_index()
g5 = px.bar(res, x="decil", y="cv_mediano",
            title="Hallazgo 5 — CV mediano por decil de tamaño (1=chico, 10=grande)",
            labels={"decil":"decil de volumen","cv_mediano":"CV mediano"})
g5.update_xaxes(dtick=1); g5.show()

## Síntesis para el modelado

| # | Hallazgo | ARIMA (`z251`) | Feature / decisión GBM-DL |
|---|---|---|---|
| 1 | Métrica ponderada / HC domina | priorizar high-volume | pesar por tn; foco en HC |
| 2 | Pico de octubre | exógena `X=` | `flag_cierre_anio` (+ `post_cierre`) |
| 3 | Ciclo de vida / poca historia | filtrar filas; 2ª pasada | `meses_desde_lanzamiento`, `span_meses` |
| 4 | Estacionalidad por categoría | `m=12` selectivo | dummies de mes + índice por cat |
| 5 | Clientes grandes regulares | no aplica | `cv_cliente`, `decil_tamano` |

**Validación (+2 con gap):** entrenar hasta `T`, saltear `T+1`, predecir `T+2`. Validar en `T+1`
sobreestima y rompe la simetría con la competencia.

**Artefactos:** `tb_panel` (series + flags + `flag_cierre_anio`), `tb_peso`, `tb_clientes`.